In [ ]:
# ruff: noqa: F401, F403

import os
import subprocess
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

from IPython.display import *

from pacer import (
    CoordinateSystem,
    # read_dat_file,
    DatVersion,
    GPMFSource,
    GPSSample,
    Lap,
    Laps,
    Point,
    PointInTime_GPSSample,
    RawGPSSource,
    ReferenceTrack,
    Segment,
    SequentialGPSSource,
    Vec3f,
)

In [5]:
files = [
    "/Volumes/Untitled/DCIM/100GOPRO/GH010259.MP4",
    "/Volumes/Untitled/DCIM/100GOPRO/GH020259.MP4",
    "/Volumes/Untitled/DCIM/100GOPRO/GH030259.MP4",
    # "/Volumes/Untitled/DCIM/100GOPRO/GH040257.MP4",
    # "/Volumes/Untitled/DCIM/100GOPRO/GH050257.MP4",
    # "/Volumes/Untitled/DCIM/100GOPRO/GH060257.MP4",
    # "/Volumes/Untitled/DCIM/100GOPRO/GH070257.MP4",
]

single_files = [GPMFSource(f) for f in files]
intermediate = []

for i in range(len(single_files)):
    if i == 0:
        intermediate.append(single_files[i])
    else:
        intermediate.append(SequentialGPSSource(intermediate[i - 1], single_files[i]))

gpmf = intermediate[-1]

gpmf.get_total_duration()
samples = []


def on_sample(s: GPSSample, _, _2):
    if s.full_speed > 3:
        samples.append((s, gpmf.current_time_span()))


while not gpmf.is_end():
    gpmf.read_samples(on_sample)
    gpmf.next()

No payload
No payload


In [6]:
570 - single_files[0].get_total_duration()

38.468994140625

In [7]:
s1, s2 = samples[0][0], samples[531][0]
print(f"Sample 1: {s1}")
print(f"Sample 2: {s2}")

Sample 1: GPSSample(lat=51.767242, lon=0.013051, altitude=29.931000, full_speed=13.757000, ground_speed=0.000000)
Sample 2: GPSSample(lat=51.767410, lon=0.011874, altitude=24.266000, full_speed=11.156000, ground_speed=10.820000)


In [8]:
cs = CoordinateSystem(s1)

In [9]:
data = np.array(
    [cs.distance(s1, s2) for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])]
)

px.scatter(data)

In [10]:
rough_frequency = len(samples) / len(set(span for _, span in samples))

data = np.array(
    [
        cs.distance(s1, s2)
        / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
        * rough_frequency
        for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
    ]
)

px.scatter(data)

In [11]:
rough_frequency

18.15602322206096

In [12]:
di = np.round(
    np.array(
        [
            cs.distance(s1, s2)
            / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
            * rough_frequency
            for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
        ]
    )
)

px.scatter(di)

In [13]:
data = data[500:]
samples = samples[500:]

In [14]:
di.sum() / (max(span[1] for _, span in samples) - min(span[0] for _, span in samples))

np.float64(18.572553354799812)

In [15]:
di.sum() / len(set(span for _, span in samples))

np.float64(18.609918578830495)

In [16]:
floor = torch.Tensor([b for (_, (b, _)) in samples])
ceil = torch.Tensor([e for (_, (_, e)) in samples])
di = np.round(
    np.array(
        [
            cs.distance(s1, s2)
            / (0.5 * s1.full_speed + 0.5 * s2.full_speed)
            * rough_frequency
            for (s1, _), (s2, _) in zip(samples[:-1], samples[1:])
        ]
    )
)
di = torch.Tensor(np.concatenate([[1], di]).astype(np.int64)).to(torch.int64)

assert ceil.shape == floor.shape == di.shape


def loss(x):
    assert x.shape == di.shape, f"Expected {di.shape}, got {x.shape}"

    my_diffs = x[1:] - x[:-1]
    my_diffs /= di[1:]
    spacing = ((my_diffs - my_diffs.mean()) ** 2).mean()
    constraints = (((floor - x).clip(min=0) + (x - ceil).clip(min=0)) ** 2).mean()
    return spacing + constraints

In [17]:
t1 = (floor + ceil) / 2
t1.requires_grad_()

learning_curve = []

for lr in [1e-1, 1e-2, 1e-3]:
    optimizer = torch.optim.Adam([t1], lr=lr)
    for i in range(100):
        optimizer.zero_grad()
        l = loss(t1)
        l.backward()
        optimizer.step()
        learning_curve.append((lr, i, l.item()))
px.line(
    pd.DataFrame(learning_curve, columns=["lr", "iteration", "loss"]),
    y="loss",
    log_y=True,
    color="lr",
    title="Learning curve",
)

In [18]:
phase = torch.tensor(floor[0], dtype=torch.float64, requires_grad=True)
frequency = torch.tensor(rough_frequency, dtype=torch.float64, requires_grad=True)

t2 = phase + 1 / frequency * (di.long().cumsum(0).to(torch.float64) - 1)
learning_curve = []

for lr in [1e-1, 1e-2, 1e-3]:
    optimizer = torch.optim.Adam([phase, frequency], lr=lr)
    for i in range(100):
        optimizer.zero_grad()
        l = loss(t2)
        l.backward()
        optimizer.step()
        t2 = phase + 1 / frequency * (di.long().cumsum(0).float() - 1)
        learning_curve.append((lr, i, l.item()))

px.line(
    pd.DataFrame(learning_curve, columns=["lr", "iteration", "loss"]),
    y="loss",
    log_y=True,
    color="lr",
    title="Learning curve",
)

/var/folders/1h/jqy0k9wj1sx7s89lp8_kgm_r0000gn/T/ipykernel_79918/1466950217.py:1: UserWarning:

To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).



In [19]:
frequency, phase

(tensor(18.1794, dtype=torch.float64, requires_grad=True),
 tensor(103.6232, dtype=torch.float64, requires_grad=True))

In [20]:
implied_frequency = di[1:] / (t1[1:] - t1[:-1])
px.scatter(implied_frequency.detach().numpy(), title="Implied frequency from GPS data")

In [21]:
implied_frequency = di[1:] / (t2[1:] - t2[:-1])
px.scatter(implied_frequency.detach().numpy(), title="Implied frequency from GPS data")

In [22]:
laps = Laps()
laps.set_coordinate_system(cs)

for (s, span), t in zip(samples, t1):
    laps.add_point(s, t)

s = laps.pick_random_start()
laps.sectors.start_line = s
laps.update()

laps_times = pd.DataFrame(
    [dict(lap=i, lap_time=laps.lap_time(i)) for i in range(laps.laps_count())]
)
px.line(
    laps_times.loc[lambda d: (d["lap_time"] > 10) & (d["lap_time"] < 80)],
    x="lap",
    y="lap_time",
    title="Lap times",
    markers=True,
)

In [23]:
best_lap = laps_times.loc[
    lambda d: d["lap_time"] > 0.95 * np.median(d["lap_time"]), "lap_time"
].idxmin()

In [ ]:
delta_by_lap = []
reference_lap = ReferenceTrack.from_lap(laps.get_lap(best_lap), 5, cs)

lap1 = reference_lap.resample(laps.get_lap(best_lap))
t1 = (
    np.array([lap1.points[i].time for i in range(len(lap1.points))])
    - lap1.points[0].time
)

for i in range(1, laps.laps_count() - 1):
    lap0 = reference_lap.resample(laps.get_lap(i))

    t0 = (
        np.array([lap0.points[i].time for i in range(len(lap0.points))])
        - lap0.points[0].time
    )

    try:
        delta = t1 - t0

        delta_by_lap.append(
            pd.DataFrame(
                dict(
                    lat=[lap0.points[i].point.lat for i in range(len(lap0.points))],
                    lon=[lap0.points[i].point.lon for i in range(len(lap0.points))],
                    speed=[
                        lap0.points[i].point.full_speed for i in range(len(lap0.points))
                    ],
                    distance=lap0.cum_distances,
                    t=t0 + lap0.points[0].time,
                    delta=delta,
                    lap=i,
                )
            )
        )
    except ValueError as e:
        print(f"Skipping lap {i} due to error: {e}")

delta_by_lap = pd.concat(delta_by_lap)


In [25]:
laps_times

,lap,lap_time
0,0,46.568791
1,1,49.843778
2,2,45.133392
3,3,45.326917
4,4,45.464571
5,5,45.923324
6,6,45.913890
7,7,46.156502
8,8,45.445512
9,9,45.749582


In [26]:
delta_by_lap

,lat,lon,speed,distance,t,delta,lap
0,51.767574,0.010609,23.071316,0.000000,183.203981,0.000000,1
1,51.767581,0.010597,23.095569,1.089285,183.250991,0.000674,1
2,51.767588,0.010583,23.034388,2.338659,183.305393,0.001387,1
3,51.767595,0.010570,23.059854,3.597861,183.359832,0.002032,1
4,51.767603,0.010556,23.143579,4.862550,183.413695,0.003239,1
...,...,...,...,...,...,...,...
815,51.767547,0.010657,17.637244,730.881130,1423.819224,-4.197173,27
816,51.767554,0.010644,17.649227,732.129460,1423.889974,-4.212900,27
817,51.767562,0.010630,17.737531,733.379610,1423.959338,-4.227271,27
818,51.767569,0.010616,17.893231,734.652704,1424.029292,-4.242233,27


In [ ]:
fig = px.line(
    delta_by_lap.assign(
        speed=lambda d: d["speed"] * 3.6,
        local_t=lambda d: d["t"] - (d["t"] / 531).astype(int) * 531,
        lap_time=lambda d: pd.merge(d, laps_times, on="lap", how="left")[
            "lap_time"
        ].values,
    ).loc[lambda d: d["lap_time"] < np.median(d["lap_time"]) * 1.07],
    x="distance",
    y="delta",
    hover_data=["local_t", "speed", "lap_time"],
    color="lap",
)
fig.show()

In [32]:
px.line(
    delta_by_lap.assign(speed=lambda d: d["speed"] * 3.6),
    x="distance",
    y="speed",
    color="lap",
)

In [ ]:
delta_by_lap.loc[lambda d: d["lap"] == 11]

,lat,lon,distance,t,delta,lap
0,51.767365,0.011462,0.000000,570.400452,0.000000,11
1,51.767363,0.011467,0.372614,570.417373,0.001010,11
2,51.767358,0.011481,1.539253,570.470997,0.002439,11
3,51.767352,0.011494,2.670940,570.523370,0.005120,11
4,51.767346,0.011507,3.768923,570.575753,0.007791,11
...,...,...,...,...,...,...
817,51.767382,0.011401,734.369636,615.529414,-0.165963,11
818,51.767376,0.011416,735.561060,615.584400,-0.166017,11
819,51.767371,0.011429,736.676864,615.636122,-0.162868,11
820,51.767365,0.011443,737.814293,615.690804,-0.162680,11


In [31]:
ddelta = delta_by_lap["delta"].diff()
ddelta = (
    ddelta.clip(
        lower=delta.mean() - 2 * ddelta.std(), upper=ddelta.mean() + 2 * ddelta.std()
    )
    .rolling(20)
    .mean()
)


px.scatter_map(
    delta_by_lap.assign(
        ddelta=ddelta, local_t=lambda d: d["t"].where(lambda d: d < 531, d["t"] - 531)
    ).loc[lambda d: d["lap"].isin([2, 23])],
    lat="lat",
    lon="lon",
    color="speed",
    hover_data=["distance", "delta", "lap", "local_t"],
    # map_style="satellite",
    zoom=17,
).update_layout(height=800)

In [44]:
px.histogram(ddelta, nbins=1_000)

In [45]:
px.histogram(delta_by_lap["delta"], nbins=200)

In [91]:
lap = laps.get_lap(1)


def build_lap_df(lap):
    return pd.DataFrame(
        [
            dict(
                lat=s.point.latitude,
                lon=s.point.longitude,
                time=s.time - lap.points[0].time,
                distance=lap.cum_distances[i],
                i_point=i,
            )
            for i in range(lap.count())
            if (s := lap.points[i]) is not None
        ]
    )


all_laps = pd.concat(
    [build_lap_df(laps.get_lap(i)).assign(i_lap=i) for i in range(laps.laps_count())]
)


AttributeError: 'pacer._pacer.GPSSample' object has no attribute 'latitude'

In [ ]:
fig = px.scatter_map(
    all_laps,
    lat="lat",
    lon="lon",
    color="distance",
    hover_data=["i_lap", "i_point"],
    # map_style="basic",
    zoom=17,
)
fig.update_layout(height=800)


In [ ]:
start_line = laps.sectors.start_line
p1, p2 = start_line.first, start_line.second

In [ ]:
s1, s2 = map(lambda p: getattr(cs, "global")(Vec3f(p.x, p.y, 0)), (p1, p2))

In [ ]:
def add_segment(fig: go.Figure, s1: GPSSample, s2: GPSSample, name: str):
    fig.add_trace(
        go.Scattermap(
            mode="lines",
            lon=[s1.longitude, s2.longitude],
            lat=[s1.latitude, s2.latitude],
            name=name,
        )
    )
    return fig


add_segment(fig, s1, s2, "start_line")

In [ ]:
Point(1, 2)

Point(x=1.0, y=2.0)

In [ ]:
start_line

Segment(first=Point(x=83.9594233828202, y=-26.0512637901178), second=Point(x=75.5699800474182, y=-31.49343619845611))

In [ ]:
def to_point(s: GPSSample):
    v = getattr(cs, "local")(s)
    return Point(v[0], v[1])


first_seg = Segment(
    to_point(laps.get_lap(0).points[0].point), to_point(laps.get_lap(0).points[1].point)
)

add_segment(
    fig, laps.get_lap(0).points[0].point, laps.get_lap(0).points[1].point, "first_one"
)

In [ ]:
ratio, _ = (
    start_line.intersects(first_seg.first, first_seg.second),
    start_line.intersects(first_seg.second, first_seg.first),
)

In [ ]:
df = pd.DataFrame(
    dict(
        x=[
            start_line.first.x,
            start_line.second.x,
            None,
            first_seg.first.x,
            first_seg.second.x,
        ],
        y=[
            start_line.first.y,
            start_line.second.y,
            None,
            first_seg.first.y,
            first_seg.second.y,
        ],
        name=[1, 1, None, 2, 3],
    )
)

fig = px.line(df, x="x", y="y", hover_data="name")

In [ ]:
x, y = start_line.first, start_line.second
a, b = first_seg.first, first_seg.second

In [ ]:
res = (1 - ratio) * a + ratio * b
fig.add_trace(go.Scatter(x=[res.x], y=[res.y]))

In [ ]:
def rot(x):
    return Point(-x.y, x.x)

In [ ]:
n = rot(x - y)
n.scalar(a - x), n.scalar(b - x), n.scalar(a - y), n.scalar(b - y)

(-4.940411947284312,
 0.8100397188352676,
 -4.940411947284313,
 0.8100397188352667)

In [ ]:
a

Point(x=79.49651065482897, y=-29.535207990925674)

In [ ]:
dir(start_line.first)

['__add__',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__sub__',
 '__subclasshook__',
 'scalar',
 'x',
 'y']

In [ ]:
start_line.first.scalar(start_line.first)

7727.85311983796

In [ ]:
start_line.intersects(first_seg.first, first_seg.second, None)

TypeError: intersects(): incompatible function arguments. The following argument types are supported:
    1. intersects(self, fst: _pacer_geometry_impl.Point, snd: _pacer_geometry_impl.Point) -> float | None

Invoked with types: _pacer_laps.Segment, _pacer_geometry_impl.Point, _pacer_geometry_impl.Point, NoneType

In [ ]:
cs.global_()

AttributeError: '_pacer_geometry_impl.CoordinateSystem' object has no attribute 'global_'

In [ ]:
px.scatter_map(
    points_df,
    lat="latitude",
    lon="longitude",
    color="speed",
    map_style="outdoors",
).update_layout(height=800)

NameError: name 'points_df' is not defined

In [ ]:
lap = laps.get_lap(1)
dir(lap)

In [ ]:
lap.points

In [ ]:
delta_by_lap

In [ ]:
px.scatter(delta[1:] - delta[:-1])

In [ ]:
c = 0.1
noise = np.round((delta[1:] - delta[:-1]) / c) * c
px.scatter(noise)

In [ ]:
px.scatter(np.cumsum(noise), title="Cumulative noise")

In [ ]:
px.line(delta - np.concatenate([[0], np.cumsum(noise)]))

In [ ]:
laps = Laps()
laps.set_coordinate_system(cs)

for (s, span), t in zip(samples, t2):
    laps.add_point(s, t)
s = laps.pick_random_start()
laps.sectors.start_line = s
laps.update()
reference_lap = ReferenceTrack.from_lap(laps.get_lap(1), 5, cs)
lap0 = reference_lap.resample(laps.get_lap(3))
lap1 = reference_lap.resample(laps.get_lap(4))
px.line(
    [
        lap1.points[i].time
        - lap0.points[i].time
        - lap1.points[0].time
        + lap0.points[0].time
        for i in range(len(lap0.points))
    ]
)